# Thesis Notebook — SRQ2 + SRQ3 (Agentic Layer)

**Owner**: Enrico Manfron  
**Institution**: Copenhagen Business School (MSc 2026)  
**Created**: 2026-04-17

---

## SRQs addressed in this notebook

- **SRQ2**: *How can a multi-agent architecture coordinate predictive models and heterogeneous data signals to generate actionable managerial recommendations?*
- **SRQ3**: *To what extent does additional contextual information improve the predictive and decision-support capabilities of AI systems?* — answered via **System A vs System B** ablation (same architecture, Indeks contextual signal toggled on/off).

## Architecture (3 agents + orchestrator, LangGraph)

```
User query
    │
    ▼
┌────────────────────────────────┐
│       ORCHESTRATOR             │
│       (LangGraph state graph)  │
└────────────────────────────────┘
    │
    ▼
┌──────────────┐   ┌──────────────┐   ┌────────────────────┐
│ FORECASTING  │──▶│  ANOMALY     │──▶│  RECOMMENDATION    │
│   AGENT      │   │   AGENT      │   │     AGENT          │
│   (Haiku)    │   │   (Haiku)    │   │   (Sonnet 4.6)     │
│              │   │              │   │                    │
│ Wraps LGBM   │   │ Detects      │   │ Synthesizes NL     │
│ from SRQ1    │   │ deviations   │   │ recommendations    │
└──────────────┘   └──────────────┘   └────────────────────┘
                                              ▲
                                              │ (only System B)
                                       ┌──────┴──────┐
                                       │ INDEKS      │
                                       │ CONTEXT     │
                                       │ PROVIDER    │
                                       │ (Haiku)     │
                                       └─────────────┘
```

## Model selection rationale (ties to SRQ1 efficiency)

Right-size LLM by task complexity:
- **Haiku 4.5**: tool-use agents (Forecasting, Anomaly, Indeks Context) — fast, cheap, adequate reasoning.
- **Sonnet 4.6**: quality-critical agent (Recommendation) — business-grade NL output.
- **No Opus**: overkill for this workload, preserves Pareto efficiency at the orchestration layer.

---

## Notebook structure (10 sections)

| § | Section | Purpose |
|---:|---|---|
| 0 | Setup | Imports, API key, LightGBM loader, model config |
| 1 | Architecture overview + graph diagram | Render LangGraph schematic |
| 2 | Forecasting Agent | Wrap SRQ1 LightGBM as a LangGraph node (Haiku) |
| 3 | Anomaly Agent | Statistical + LLM-reasoning node (Haiku) |
| 4 | Indeks Context Provider | Extract regional context from Indeks (Haiku) |
| 5 | Recommendation Agent — System A | NL recs using Nielsen signals only (Sonnet) |
| 6 | Recommendation Agent — System B | System A + Indeks contextual layer (Sonnet) |
| 7 | Orchestrator graph compile + smoke test | End-to-end run on 1 query |
| 8 | A/B comparison | Run 20 sample queries in A and B; metrics |
| 9 | Final figures + conclusion | Thesis-ready outputs |

**Rules of engagement** (same as SRQ1 notebook):
1. Run each cell, inspect output, fill in an "Observations + Decisions" markdown block.
2. No section skipped — every design choice is logged in the notebook itself.
3. Final deliverable = notebook + outputs/figures + committable .ipynb.

---

# §0 — Setup

**Why**: fix imports (LangGraph + anthropic + LightGBM loader), API key, model constants, paths. All downstream cells depend on this.

**Output**: printed block confirming API key is set, LightGBM model loads, LangGraph version OK.

In [4]:
# §0 — Setup
import sys
print(sys.executable)
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

import os
import json
import pickle
import warnings
from pathlib import Path
from typing import TypedDict, Literal, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# LangGraph + Anthropic
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

import anthropic
import langgraph

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["figure.dpi"] = 110

# ──────────────────────────────────────────────────────────────────────────
# Paths — anchor to project root
# ──────────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CLAUDE.md").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRQ1_OUTPUTS = PROJECT_ROOT / "docs" / "thesis" / "analysis" / "outputs"
SRQ1_PIPELINES = SRQ1_OUTPUTS / "pipelines"
OUTPUT_DIR = PROJECT_ROOT / "docs" / "thesis" / "analysis" / "outputs_agentic"
FIGURE_DIR = PROJECT_ROOT / "docs" / "thesis" / "analysis" / "figures_agentic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────
# API key — load from .env if present
# ──────────────────────────────────────────────────────────────────────────
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.startswith("ANTHROPIC_API_KEY=") and "=" in line:
            os.environ["ANTHROPIC_API_KEY"] = line.split("=", 1)[1].strip().strip('"').strip("'")
            break

api_ok = bool(os.environ.get("ANTHROPIC_API_KEY"))

# ──────────────────────────────────────────────────────────────────────────
# Model constants (right-sized per task per SRQ1 efficiency principle)
# ──────────────────────────────────────────────────────────────────────────
MODEL_HAIKU  = "claude-haiku-4-5-20251001"
MODEL_SONNET = "claude-sonnet-4-6"

llm_haiku  = ChatAnthropic(model=MODEL_HAIKU,  temperature=0, max_tokens=1024)
llm_sonnet = ChatAnthropic(model=MODEL_SONNET, temperature=0.3, max_tokens=2048)

# ──────────────────────────────────────────────────────────────────────────
# Load SRQ1 artefacts (LightGBM + pipeline + feature matrix)
# ──────────────────────────────────────────────────────────────────────────
fm_path = SRQ1_OUTPUTS / "feature_matrix_split.parquet"
lgbm_path = SRQ1_PIPELINES / "model_lightgbm.pkl"
pipe_tree_path = SRQ1_PIPELINES / "pipe_tree.pkl"
feature_info_path = SRQ1_PIPELINES / "feature_info.json"

srq1_artefacts_ok = all(p.exists() for p in [fm_path, lgbm_path, pipe_tree_path, feature_info_path])

if srq1_artefacts_ok:
    fm = pd.read_parquet(fm_path)
    with open(lgbm_path, "rb") as f:
        lgbm_model = pickle.load(f)
    with open(pipe_tree_path, "rb") as f:
        pipe_tree = pickle.load(f)
    feature_info = json.loads(feature_info_path.read_text())
else:
    fm, lgbm_model, pipe_tree, feature_info = None, None, None, None

# ──────────────────────────────────────────────────────────────────────────
# Print diagnostic summary
# ──────────────────────────────────────────────────────────────────────────
print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"FIGURE_DIR      : {FIGURE_DIR.relative_to(PROJECT_ROOT)}")
print()
print(f"ANTHROPIC_API_KEY set? {api_ok}")
print(f"langgraph version      {langgraph.__version__}")
print(f"anthropic version      {anthropic.__version__}")
print()
print("Model assignments:")
print(f"  Haiku  (tool-use agents): {MODEL_HAIKU}")
print(f"  Sonnet (quality-critical): {MODEL_SONNET}")
print()
print("SRQ1 artefacts:")
if srq1_artefacts_ok:
    print(f"  ✅ feature_matrix   : {fm.shape[0]:,} rows × {fm.shape[1]} cols")
    print(f"  ✅ LightGBM model   : loaded ({feature_info['output_shape_tree'][1]} features)")
    print(f"  ✅ pipe_tree        : loaded")
    print(f"  ✅ feature_info     : {len(feature_info['numeric_feats'])} numeric, {len(feature_info['cat_feats'])} cat")
else:
    print("  ⚠️  One or more SRQ1 artefacts missing. Re-run SRQ1 notebook first.")

/usr/local/bin/python3
Python executable: /usr/local/bin/python3
Python version: 3.13.3 (v3.13.3:6280bb54784, Apr  8 2025, 10:47:54) [Clang 15.0.0 (clang-1500.3.9.4)]


ModuleNotFoundError: No module named 'langchain_anthropic'

### §0 — Observations + Decisions

_To be filled in after running the cell above._

- API key set? _..._
- LangGraph + Anthropic versions OK? _..._
- SRQ1 artefacts loaded cleanly? _..._
- Any unexpected warnings? _..._

---

# §1 — Architecture overview + graph diagram

**Why**: before writing agents, formalise the state schema (what flows between agents) and the graph topology. Rendering the LangGraph diagram as PNG gives a thesis-ready architecture figure.

**Output**: `ForecastState` TypedDict definition + a rendered PNG of the empty graph topology (nodes exist, bodies are stubs to be filled in §2-§6).

In [ ]:
# TODO §1: define ForecastState TypedDict + render empty graph topology as PNG

### §1 — Observations + Decisions

_To be filled in._

---

# §2 — Forecasting Agent (Haiku)

**Why**: wraps the SRQ1 LightGBM model as a LangGraph node. Haiku parses the user's natural-language query (e.g., *"What will HARBOE sell in DVH in Oct 2025?"*) into structured args, calls the LightGBM tool, and returns a forecast + SHAP top-drivers.

**Output**: a callable node function `forecasting_agent(state)` + unit test on a sample query.

In [ ]:
# TODO §2: forecasting_agent node + smoke test

### §2 — Observations + Decisions

_To be filled in._

---

# §3 — Anomaly Agent (Haiku)

**Why**: given a forecast from §2, this agent decides whether the predicted value is anomalous vs the brand-channel's historical pattern. Statistical core (z-score vs rolling window) + LLM reasoning layer for interpretation ("driven by seasonality" vs "structural shift").

**Output**: `anomaly_agent(state)` node producing `anomalies: list[dict]` with type, severity, rationale.

In [ ]:
# TODO §3: anomaly_agent node + smoke test

### §3 — Observations + Decisions

_To be filled in._

---

# §4 — Indeks Context Provider (Haiku)

**Why**: in System B only, this node reads the Indeks Danmark survey data and extracts 3-5 contextual facts relevant to the queried brand (e.g., *"brand awareness 67% in Jutland"*, *"purchase intent up 4pp YoY"*). Used downstream by the Recommendation Agent to ground recommendations.

**Output**: `indeks_context(state)` node producing `context: dict` with brand-region-metric triples.

In [ ]:
# TODO §4: indeks_context node + smoke test

### §4 — Observations + Decisions

_To be filled in._

---

# §5 — Recommendation Agent — System A (Sonnet 4.6)

**Why**: System A synthesises forecast + anomalies into natural-language managerial recommendations using **only Nielsen signals**. This is the baseline for SRQ3's A/B ablation. Specificity requirement: every recommendation must include a verb, a number, and a deadline.

**Output**: `recommendation_agent_A(state)` node producing `recommendations: str` (markdown-formatted, manager-facing).

In [ ]:
# TODO §5: recommendation_agent (System A variant) + smoke test

### §5 — Observations + Decisions

_To be filled in._

---

# §6 — Recommendation Agent — System B (Sonnet 4.6 + Indeks context)

**Why**: System B extends System A by injecting Indeks contextual facts (from §4) into the prompt. Same Sonnet model, same output format — only the input context differs. This isolates the SRQ3 effect (Δ = B − A) cleanly.

**Output**: `recommendation_agent_B(state)` node. Identical to System A plumbing, prompt augmented with context.

In [ ]:
# TODO §6: recommendation_agent (System B variant) + side-by-side smoke test vs System A

### §6 — Observations + Decisions

_To be filled in._

---

# §7 — Orchestrator graph compile + smoke test

**Why**: assemble the 4 nodes into a `StateGraph` with conditional routing (System A skips Indeks node; System B includes it). Compile, visualise final topology, run end-to-end on one concrete query.

**Output**: compiled `app` ready to invoke; PNG of final graph; 1 end-to-end smoke test printing all intermediate state transitions.

In [ ]:
# TODO §7: compile graph + smoke test on e.g. 'HARBOE DVH_EXCL_HD 2025-10'

### §7 — Observations + Decisions

_To be filled in._

---

# §8 — A/B comparison (SRQ3 answer)

**Why**: run the orchestrator over **~20 sample queries** in both System A and System B modes. Measure:
- **Specificity rate**: % of recommendations containing verb + number + deadline.
- **Quantitative groundedness**: # numeric references per recommendation.
- **Coverage**: # distinct recommendation categories (distribution, pricing, promo, assortment).
- **LLM-as-judge score** (optional): Sonnet rates pairs of A vs B recommendations blinded.
- **Latency + token cost** per run (SRQ1 continuity).

**Output**: `outputs_agentic/ab_comparison.csv` + summary table + Δ System B − A for each metric.

In [ ]:
# TODO §8: define 20 sample queries, run both systems, collect metrics, save CSV

### §8 — Observations + Decisions

_To be filled in._

---

# §9 — Final figures + conclusion

**Why**: produce 2-3 publication-ready PNGs for the thesis:
1. Agent graph architecture diagram (final version from §7).
2. A/B metric comparison bar chart (from §8).
3. Example side-by-side: same query, System A rec vs System B rec (text box figure).

**Output**: `figures_agentic/final_01..03.png` + notebook wrap-up summary of SRQ2 + SRQ3 claims.

In [ ]:
# TODO §9: render final figures + wrap-up print

### §9 — Observations + Decisions

_To be filled in._

---

## End of notebook

When every Observations+Decisions block above is filled, this notebook **is** the methodology + evidence artefact for SRQ2 (architecture) and SRQ3 (contextual information).